### Util functions & Settings

In [ ]:
from IPython.display import Video, display
import torch

def make_env(env_id, required_experience_length, mode="change", render_mode=None, **physics_kwargs):
    family = env_id.split("-")[0]
    def _make():
        env = gym.make(env_id, render_mode=render_mode)
        if family == "Ant":
            env = CleanObservationWrapper(env)
        env = PhysicsWrapper(env, mode=mode, required_experience_length=required_experience_length,
                             **PHYSICS_SPECS[family], **physics_kwargs)
        if family == "Ant":
            env = BodyFrameWrapper(env)
        if family == "Walker2d":
            env = LimitAngleTorqueWrapper(env)
        return env
    return _make



class EmbeddingVisualizer:

    # training ranges per parameter, used to normalise the 3-D parameter panel to [0, 1]
    PARAM_RANGES = {"mass": (0.5, 2.0), "damping": (0.5, 2.0), "torque_scale": (0.5, 1.5)}
    PARAM_AXES = ("mass", "damping", "torque_scale")

    def __init__(self, agent, env_id="Hopper-v5"):

        self.agent = agent
        env_id = env_id or agent.env_ID
        self.env = make_env(env_id, None, mode="nominal")()
        self.friction_range = PHYSICS_SPECS[env_id.split("-")[0]]["friction_range"]

        # default physics, to scale from and restore
        self.model = self.env.unwrapped.model
        self.default_mass = self.model.body_mass.copy()
        self.default_friction = self.model.geom_friction.copy()
        self.default_damping = self.model.dof_damping.copy()
        self.torque_scale = 1.0

        self.n_past = agent.config.n_past
        self.obs_dim = agent.obs_dim
        self.action_dim = agent.action_dim
        self.device = agent.device

   
    def rollout(self, seed):
        """One deterministic episode; returns encoder means, shape (T, latent_dim)."""
        self.agent.encoder.eval()
        self.agent.actor.eval()

        history = HistoryQueue(1, self.n_past, self.obs_dim, self.action_dim)
        latents = []

        obs, _ = self.env.reset(seed=seed)
        history.reset_done(np.ones(1, dtype=bool), obs[None])   # pad with initial obs, as in training

        done = False
        while not done:
            with torch.no_grad():
                history_t = torch.as_tensor(history.get(), device=self.device)
                obs_t = torch.as_tensor(obs, dtype=torch.float32, device=self.device).reshape(1, -1)
                z = self.agent.encoder(history_t).loc                     # mean, no sampling
                action = self.agent.actor.predict(torch.cat([obs_t, z], dim=-1), deterministic=True)
            z, action = z.cpu().numpy()[0], action.cpu().numpy()[0]
            latents.append(z)

            env_action = self.agent._toEnvAction(action) * self.torque_scale
            next_obs, _, terminated, truncated, _ = self.env.step(env_action)

            # newest (obs, normalized action) pair goes to slot 0
            history.append(obs[None], action[None])

            obs = next_obs
            done = terminated or truncated

        return np.array(latents)

   
    def set_dynamics(self, param_name, scalar):
        """Scale ONE parameter group from default: 'mass', 'friction', 'damping', 'torque_scale'."""
        self.reset_dynamics()
        if param_name == "mass":
            self.model.body_mass[:] = self.default_mass * scalar
        elif param_name == "friction":
            friction = self.default_friction.copy()
            friction[:, 0] *= scalar
            self.model.geom_friction[:] = friction
        elif param_name == "damping":
            self.model.dof_damping[:] = self.default_damping * scalar
        elif param_name == "torque_scale":
            self.torque_scale = scalar
        else:
            raise ValueError(f"Unknown parameter: {param_name}")

    # def randomize_dynamics(self, rng):
    #     """Random draw from the training ranges (Xu's HopperDynamicsWrapper)."""
    #     self.model.body_mass[:] = self.default_mass * rng.uniform(0.5, 2.0, size=self.default_mass.shape)
    #     self.model.dof_damping[:] = self.default_damping * rng.uniform(0.5, 2.0, size=self.default_damping.shape)
    #     self.torque_scale = float(rng.uniform(0.5, 1.5))

    def randomize_dynamics(self, rng):
        """Random draw from Xu's training ranges (friction only for HalfCheetah/Ant)."""
        if self.friction_range is not None:
            fric = self.default_friction.copy()
            fric[:, 0] *= rng.uniform(*self.friction_range, size=fric.shape[0])
            self.model.geom_friction[:] = fric
        self.model.body_mass[:] = self.default_mass * rng.uniform(0.5, 2.0, size=self.default_mass.shape)
        self.model.dof_damping[:] = self.default_damping * rng.uniform(0.5, 2.0, size=self.default_damping.shape)
        self.torque_scale = float(rng.uniform(0.5, 1.5))

    def reset_dynamics(self):
        self.model.body_mass[:] = self.default_mass
        self.model.geom_friction[:] = self.default_friction
        self.model.dof_damping[:] = self.default_damping
        self.torque_scale = 1.0

    def current_param_scales(self):
        """
        Effective (mass, damping, torque_scale) multipliers of the current model.
        Mass and damping are randomised per body / per joint, so each is reduced to one number:
        total current value / total default value (mass-weighted and damping-weighted mean scale).
        Bodies/DOFs with zero default (world body, free root joints) contribute nothing.
        """
        def ratio(cur, default):
            tot = default.sum()
            return float(cur.sum() / tot) if tot > 0 else 1.0
        return np.array([ratio(self.model.body_mass, self.default_mass),
                         ratio(self.model.dof_damping, self.default_damping),
                         float(self.torque_scale)])


    def current_params(self):
            return np.array([float(self.model.body_mass.sum()/len(self.model.body_mass)),
                             float(self.model.dof_damping.sum()/len(self.model.dof_damping)),
                             float(self.torque_scale)])

    @classmethod
    def normalize_params(cls, params):
        """Map raw scales to [0, 1] per axis using the training range; OOD values fall outside [0, 1]."""
        lo = np.array([cls.PARAM_RANGES[k][0] for k in cls.PARAM_AXES])
        hi = np.array([cls.PARAM_RANGES[k][1] for k in cls.PARAM_AXES])
        return (np.asarray(params, dtype=float) - lo) / (hi - lo)

    
    def plot_separation(self, num_trajs=10, n_latents_per_traj=100, projection_method="pca", seed=0,
                        show_params=True):
        """One trajectory per random physics draw, coloured by trajectory (Xu: visualizeSeparationAcrossDynamics).
        show_params: add a 3-D panel with each trajectory's normalised (mass, damping, torque) in the same colour."""
        rng = np.random.default_rng(seed)
        all_latents, colors, params = [], [], []
        for i in range(num_trajs):
            self.randomize_dynamics(rng)
            params.append(self.current_param_scales()) #getting the scales of the parameters instead of the raw values
            # params.append(self.current_params()) # getting the raw values of the parameters instead of the scales
            latents = self._subsample(self.rollout(seed=seed + i), n_latents_per_traj)
            all_latents.append(latents)
            colors.extend([i] * len(latents))
        self.reset_dynamics()

        colors = np.array(colors)
        norm = plt.Normalize(vmin=colors.min(), vmax=colors.max())
        fig, ax, ax_p, pts = self._figure(np.concatenate(all_latents), projection_method, show_params)
        self._scatter(ax, pts, c=colors, cmap="jet", norm=norm, s=20, marker="o", alpha=0.8)

        if ax_p is not None:
            ids = np.arange(num_trajs)
            self._plot_params(ax_p, np.array(params), c=ids, cmap="jet", norm=norm,
                              labels=[str(i) for i in ids])

        cbar = fig.colorbar(cm.ScalarMappable(cmap="jet", norm=norm),
                            ax=[a for a in (ax, ax_p) if a is not None], pad=0.02, shrink=0.8)
        cbar.set_ticks(np.arange(num_trajs))
        cbar.set_label("Trajectory ID", fontsize=14)
        self.last_params = np.array(params)          # raw scales, one row per trajectory
        return fig, (ax, ax_p)

    def plot_physics_scan(self, param_name, scan_values, train_range, seed=1012,
                          n_latents_per_traj=100, projection_method="tsne", show_params=False):
        """
        Scale one parameter over scan_values, same seed (same start) each time
        (Xu: visualizeLatentSemanticStructure).
        train_range: [min, max) seen in training -> (0.5, 2.0) mass/damping, (0.5, 1.5) torque_scale
        show_params: add a 3-D panel with each scan's normalised (mass, damping, torque) in the same colour.
        """
        all_latents, values, ood, params = [], [], [], []
        for value in scan_values:
            self.set_dynamics(param_name, value)
            params.append(self.current_param_scales())
            latents = self._subsample(self.rollout(seed=seed), n_latents_per_traj)
            all_latents.append(latents)
            values.extend([value] * len(latents))
            ood.extend([value < train_range[0] or value >= train_range[1]] * len(latents))
        self.reset_dynamics()

        values, ood = np.array(values), np.array(ood, dtype=bool)
        norm = plt.Normalize(vmin=values.min(), vmax=values.max())
        fig, ax, ax_p, pts = self._figure(np.concatenate(all_latents), projection_method, show_params)
        for mask, marker, label in [(~ood, "o", "In-Distribution"), (ood, "^", "Out-Of-Distribution")]:
            if mask.any():
                self._scatter(ax, pts[mask], c=values[mask], cmap="RdBu_r", norm=norm,
                              s=20, marker=marker, label=label, alpha=0.8)

        if ax_p is not None:
            scan = np.array(scan_values, dtype=float)
            scan_ood = (scan < train_range[0]) | (scan >= train_range[1])
            self._plot_params(ax_p, np.array(params), c=scan, cmap="RdBu_r", norm=norm,
                              ood=scan_ood, labels=[f"{v:.2f}" for v in scan])

        cbar = fig.colorbar(cm.ScalarMappable(cmap="RdBu_r", norm=norm),
                            ax=[a for a in (ax, ax_p) if a is not None], pad=0.02, shrink=0.8)
        cbar.set_ticks(scan_values)
        cbar.set_label(f"{param_name} value", fontsize=14)
        # ax.legend(fontsize=14, loc="upper right", markerscale=2)
        self.last_params = np.array(params)
        return fig, (ax, ax_p)

    
    @staticmethod
    def _subsample(latents, n):
        idx = np.linspace(0, len(latents) - 1, min(len(latents), n), dtype=int)
        return latents[idx]

    @staticmethod
    def project(latents, method="pca", dim=2):
        """'pca' or 'tsne' to `dim` dims; returns input unchanged if it is already <= dim."""
        if latents.shape[1] <= dim:
            return latents.copy()
        if method == "pca":
            pca = PCA(n_components=dim)
            out = pca.fit_transform(latents)
            print("Cumulative explained variance:", np.cumsum(pca.explained_variance_ratio_))
            return out
        if method == "tsne":
            # Xu uses perplexity = N - 2; slow for large N, use 'pca' then
            return TSNE(n_components=dim, perplexity=len(latents) - 2, random_state=42).fit_transform(latents)
        raise ValueError("method must be 'pca' or 'tsne'")

    def _figure(self, latents, projection_method, show_params=False):
        """Latent panel ('3d' plots raw z or PCA to 3 dims; otherwise 2-D projection),
        plus an optional 3-D parameter panel to its right. Returns fig, ax, ax_params (or None), pts."""
        assert projection_method in ["pca", "tsne", "3d"]
        ncols = 2 if show_params else 1
        fig = plt.figure(figsize=(9 * ncols, 8.5))
        if projection_method == "3d":
            assert latents.shape[1] >= 3, "'3d' needs latent_dim >= 3"
            pts = self.project(latents, "pca", dim=3)
            ax = fig.add_subplot(1, ncols, 1, projection="3d")
            ax.view_init(elev=30, azim=45)
        else:
            pts = self.project(latents, projection_method, dim=2)
            ax = fig.add_subplot(1, ncols, 1)
            ax.set_xticks([])
            ax.set_yticks([])
        ax.set_title("Latent embedding", fontsize=14)
        ax_p = fig.add_subplot(1, ncols, 2, projection="3d") if show_params else None
        return fig, ax, ax_p, pts

    def _plot_params(self, ax, params, c, cmap, norm, ood=None, labels=None):
        """One point per trajectory at its normalised (mass, damping, torque_scale); [0,1]^3 = training box."""
        p = self.normalize_params(params)
        ood = np.zeros(len(p), dtype=bool) if ood is None else np.asarray(ood, dtype=bool)
        c = np.asarray(c)
        for mask, marker in [(~ood, "o"), (ood, "^")]:
            if mask.any():
                ax.scatter(p[mask, 0], p[mask, 1], p[mask, 2], c=c[mask], cmap=cmap, norm=norm,
                           s=90, marker=marker, edgecolors="k", linewidths=0.5, alpha=0.95, depthshade=False)
        if labels is not None:
            for (x, y, z), lab in zip(p, labels):
                ax.text(x, y, z, f" {lab}", fontsize=9)

        # wireframe of the training box [0, 1]^3
        for s in (0, 1):
            for t in (0, 1):
                ax.plot([0, 1], [s, s], [t, t], color="gray", lw=0.6, alpha=0.5)
                ax.plot([s, s], [0, 1], [t, t], color="gray", lw=0.6, alpha=0.5)
                ax.plot([s, s], [t, t], [0, 1], color="gray", lw=0.6, alpha=0.5)

        # axes cover the training box and any OOD points
        lo = np.minimum(p.min(axis=0), 0) - 0.05
        hi = np.maximum(p.max(axis=0), 1) + 0.05
        ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_zlim(lo[2], hi[2])
        names = [f"{k}\n[{self.PARAM_RANGES[k][0]}, {self.PARAM_RANGES[k][1]})" for k in self.PARAM_AXES]
        ax.set_xlabel(names[0], fontsize=10); ax.set_ylabel(names[1], fontsize=10); ax.set_zlabel(names[2], fontsize=10)
        ax.view_init(elev=25, azim=45)
        ax.set_title("Normalised dynamics parameters scale", fontsize=14)

    @staticmethod
    def _scatter(ax, pts, **kwargs):
        if pts.shape[1] == 3:
            return ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], **kwargs)
        return ax.scatter(pts[:, 0], pts[:, 1], **kwargs)

    def close(self):
        self.env.close()




In [7]:
@torch.no_grad()
def evaluate_in_distribution(agent, env_id, n_configs=5, episodes_per_config=10, config_seed=2026, episode_seed=0, save_dir=None):
    agent.actor.eval()
    agent.encoder.eval()

    # mode="nominal": the wrapper never resamples on its own; we set each configuration explicitly
    env = make_env(env_id, required_experience_length=None, mode="nominal")()
    physics = GlobalDynamicsIDWrapper._find_physics_wrapper(env)

    def ratio(cur, default):
        return float(cur.sum() / default.sum()) if default.sum() > 0 else 1.0

    rows = []
    for c in range(n_configs):
        # sample one dynamics configuration from the training ranges, fixed for its 10 episodes
        physics._rng = np.random.default_rng(config_seed + c)
        physics._sample_physics()
        m = physics.model
        cfg = dict(
            config=c,
            mass_scale=ratio(m.body_mass, physics.base_mass),
            damping_scale=ratio(m.dof_damping, physics.base_damping),
            torque_scale=physics.torque_scale,
            friction_scale=ratio(m.geom_friction[:, 0], physics.base_friction[:, 0]),
        )

        for e in range(episodes_per_config):
            seed = episode_seed + c * episodes_per_config + e        # different initial state per episode
            obs, _ = env.reset(seed=seed)                            # nominal mode: physics stay as set

            history = HistoryQueue(1, agent.config.n_past, agent.obs_dim, agent.action_dim)
            history.reset_done(np.ones(1, dtype=bool), obs[None])   # fresh history each episode (zero-shot)

            ep_return, ep_len, done = 0.0, 0, False
            while not done:
                action = agent.predict(obs[None], history=history.get(), deterministic=True)[0]
                next_obs, reward, terminated, truncated, _ = env.step(action)
                history.append(obs[None], agent._reverseEnvAction(action)[None])
                ep_return += reward
                ep_len += 1
                obs = next_obs
                done = terminated or truncated

            rows.append({**cfg, "episode": e, "seed": seed, "reward": ep_return,
                         "length": ep_len, "fell": bool(terminated)})
    env.close()

    df = pd.DataFrame(rows)
    per_config = (df.groupby("config")
                    .agg(mass_scale=("mass_scale", "first"), damping_scale=("damping_scale", "first"),
                         torque_scale=("torque_scale", "first"), friction_scale=("friction_scale", "first"),
                         reward_mean=("reward", "mean"), reward_std=("reward", lambda x: x.std(ddof=0)),
                         falls=("fell", "sum")))
    summary = dict(
        mean=df["reward"].mean(),
        std_episodes=df["reward"].std(ddof=0),                    # spread over all 50 episodes
        std_configs=per_config["reward_mean"].std(ddof=0),        # spread across the 5 configurations
        n_episodes=len(df),
    )
    if save_dir is not None:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        df.to_csv(Path(save_dir) / f"{agent.env_name}_ID_eval_episodes.csv", index=False)
        per_config.to_csv(Path(save_dir) / f"{agent.env_name}_ID_eval_configs.csv")
    return summary, per_config, df

In [ ]:
# Out-of-distribution (OOD) evaluation, one parameter at a time
#   for each parameter: 5 configurations sampled 25% beyond the training range (below or above),
#   all other parameters at nominal, 10 episodes per configuration, frozen policy and encoder

TRAIN_RANGES = {"mass": (0.5, 2.0), "damping": (0.5, 2.0), "torque": (0.5, 1.5)}   # Xu, all envs


def _ood_bands(train_range, margin=0.25, margin_mode="bound"):
    lo, hi = train_range
    if margin_mode == "bound":            # 25% of each bound
        return (lo * (1 - margin), lo), (hi, hi * (1 + margin))
    if margin_mode == "width":            # 25% of the range width
        w = margin * (hi - lo)
        return (max(lo - w, 1e-3), lo), (hi, hi + w)
    raise ValueError("margin_mode must be 'bound' or 'width'")


def _set_single_param(physics, param, scale):
    """All parameters nominal, then scale ONE group uniformly."""
    fric = physics.base_friction.copy()
    mass, damp, torque = physics.base_mass.copy(), physics.base_damping.copy(), 1.0
    if param == "mass":
        mass *= scale
    elif param == "damping":
        damp *= scale
    elif param == "torque":
        torque = scale
    elif param == "friction":
        fric[:, 0] *= scale
    else:
        raise ValueError(f"Unknown parameter '{param}'")
    physics._set_physics(mass, damp, torque, fric)


@torch.no_grad()
def _run_episodes(agent, env, n_episodes, seed_start):
    """Deterministic rollouts with a fresh history each episode; returns list of (return, length, fell)."""
    out = []
    for e in range(n_episodes):
        obs, _ = env.reset(seed=seed_start + e)
        history = HistoryQueue(1, agent.config.n_past, agent.obs_dim, agent.action_dim)
        history.reset_done(np.ones(1, dtype=bool), obs[None])
        ep_return, ep_len, done = 0.0, 0, False
        while not done:
            action = agent.predict(obs[None], history=history.get(), deterministic=True)[0]
            next_obs, reward, terminated, truncated, _ = env.step(action)
            history.append(obs[None], agent._reverseEnvAction(action)[None])
            ep_return += reward
            ep_len += 1
            obs = next_obs
            done = terminated or truncated
        out.append((ep_return, ep_len, bool(terminated)))
    return out


@torch.no_grad()
def evaluate_ood_single_param(agent, env_id, params=None, n_configs=5, episodes_per_config=10,
                              margin=0.25, margin_mode="bound", config_seed=20_000, episode_seed=0,
                              nominal_episodes=10, save_dir=None):
    agent.actor.eval()
    agent.encoder.eval()

    family = env_id.split("-")[0]
    train_ranges = dict(TRAIN_RANGES)
    if PHYSICS_SPECS[family]["friction_range"] is not None:
        train_ranges["friction"] = PHYSICS_SPECS[family]["friction_range"]
    if params is None:
        params = list(train_ranges)                       # friction only where it was trained
    for p in params:
        if p == "friction" and p not in train_ranges:
            train_ranges[p] = (1.0, 1.0)                  # never randomized in training: nominal only

    env = make_env(env_id, required_experience_length=None, mode="nominal")()
    physics = GlobalDynamicsIDWrapper._find_physics_wrapper(env)

    # nominal reference (all parameters at 1.0), same episode seeds as config 0
    _set_single_param(physics, "mass", 1.0)
    nominal = _run_episodes(agent, env, nominal_episodes, episode_seed)
    nominal_mean = float(np.mean([r for r, _, _ in nominal]))

    rows = []
    for p_idx, param in enumerate(params):
        below, above = _ood_bands(train_ranges[param], margin, margin_mode)
        rng = np.random.default_rng(config_seed + 100 * p_idx)
        for c in range(n_configs):
            side, band = ("below", below) if c % 2 == 0 else ("above", above)
            scale = float(rng.uniform(*band))
            _set_single_param(physics, param, scale)
            # same episode seeds for every parameter, so the parameters are compared on the same start states
            for e, (ret, length, fell) in enumerate(
                    _run_episodes(agent, env, episodes_per_config, episode_seed + c * episodes_per_config)):
                rows.append(dict(param=param, config=c, side=side, scale=scale,
                                 train_low=train_ranges[param][0], train_high=train_ranges[param][1],
                                 episode=e, reward=ret, length=length, fell=fell))
    _set_single_param(physics, "mass", 1.0)
    env.close()

    df = pd.DataFrame(rows)
    per_config = (df.groupby(["param", "config"])
                    .agg(side=("side", "first"), scale=("scale", "first"),
                         reward_mean=("reward", "mean"), reward_std=("reward", lambda x: x.std(ddof=0)),
                         falls=("fell", "sum"))
                    .reset_index())
    per_param = (df.groupby("param", sort=False)
                   .agg(train_range=("train_low", lambda s: f"[{s.iloc[0]:.2f}, {df.loc[s.index, 'train_high'].iloc[0]:.2f}]"),
                        reward_mean=("reward", "mean"),
                        std_episodes=("reward", lambda x: x.std(ddof=0)),
                        falls=("fell", "sum"),
                        n_episodes=("reward", "size")))
    per_param["std_configs"] = per_config.groupby("param")["reward_mean"].std(ddof=0)
    per_param["below_mean"] = df[df.side == "below"].groupby("param")["reward"].mean()
    per_param["above_mean"] = df[df.side == "above"].groupby("param")["reward"].mean()
    per_param["pct_of_nominal"] = 100 * per_param["reward_mean"] / nominal_mean

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_dir / f"{agent.env_name}_OOD_eval_episodes.csv", index=False)
        per_config.to_csv(save_dir / f"{agent.env_name}_OOD_eval_configs.csv", index=False)
        per_param.to_csv(save_dir / f"{agent.env_name}_OOD_eval_params.csv")
    return per_param, per_config, df, nominal_mean



## Result analysis

### Model checkpointing

In [ ]:
ENV_ID = "Walker2d-v5"          # or "Walker2d-v5", "Ant-v5", "Hopper-v5" "HalfCheetah-v5"
TRAIN_SWITCH = {"HalfCheetah-v5": 5000}.get(ENV_ID, 2000)

NUM_ENVS = 16

model = 'ldr'

if model == 'ldr':
    from ldr import *
elif model == 'ldr_fn':
    from ldr_fn import *
elif model == 'kmeans_fn':
    from kmeans_fn import *
elif model == 'dpgmm_hard':
    from dpgmm_hard import *
elif model == 'dpgmm_soft':
    from dpgmm_soft import * 


train_env = GlobalDynamicsIDWrapper(makeVecEnv([make_env(ENV_ID, TRAIN_SWITCH) for _ in range(NUM_ENVS)]))
eval_env  = GlobalDynamicsIDWrapper(makeVecEnv([make_env(ENV_ID, 1) for _ in range(NUM_ENVS)]))

config = LDRConfig(**ENV_PRESETS[ENV_ID])

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)

agent = LDRAgent(env=train_env, env_ID=ENV_ID, config=config)

In [2]:
env_name = ENV_ID.split("-")[0].lower()                     # "halfcheetah", "ant", ...
checkpoint_path = Path(f"/network/rit/lab/dahlin_lab/Gobinda/uclr/checkpoints/{model}/{env_name}_{model}_results/{env_name}_{model}_best.pt")

if checkpoint_path.exists():
    print("end-to-end test: loading checkpoint for inference")
    inference_env = makeVecEnv([make_env(ENV_ID, None, mode="nominal")])   # 1 env, nominal physics
    ckpt_cfg = torch.load(checkpoint_path, map_location="cpu", weights_only=True)["config"]
    inference_config = LDRConfig(**{**ckpt_cfg, "buffer_size": 1})
    inference_agent = LDRAgent(inference_env, env_ID=ENV_ID, config=inference_config)
    inference_agent.load(str(checkpoint_path))
    inference_env.close()
else:
    inference_agent = agent
print("Inference policy ready")

end-to-end test: loading checkpoint for inference
Inference policy ready


### Video

In [6]:
VIDEO_DIR = Path("./videos")
VIDEO_DIR.mkdir(exist_ok=True)

# Headless rendering environment
env = make_env(ENV_ID, None, mode="nominal", render_mode="rgb_array")()

obs, info = env.reset(seed=42)
history = HistoryQueue(1, inference_agent.config.n_past,
                       inference_agent.obs_dim, inference_agent.action_dim)
history.reset_done(np.ones(1, dtype=bool), obs[None])     # pad with initial obs

frames, total_reward = [], 0.0
fps = round(1.0 / env.unwrapped.dt)

for step in range(1000):
    action = inference_agent.predict(obs[None], history=history.get(),
                                     deterministic=True)[0]          # env-scale action

    next_obs, reward, terminated, truncated, info = env.step(action)

    # store (obs, normalized action), newest first
    history.append(obs[None], inference_agent._reverseEnvAction(action)[None])

    frames.append(env.render())
    total_reward += reward
    obs = next_obs
    if terminated or truncated:
        break

env.close()
video_path = VIDEO_DIR / f"{inference_agent.env_name}_ldr.mp4"


imageio.mimsave(
    str(video_path),
    frames,
    fps=fps,
    macro_block_size=16
)

print(f"Episode reward: {total_reward:.2f}")
print(f"Video saved: {video_path.resolve()}")

# Display inside Jupyter
display(Video(
    filename=str(video_path),
    embed=True
))

Episode reward: 3297.80
Video saved: /network/rit/lab/dahlin_lab/Gobinda/uclr/videos/hopper_ldr.mp4


### Latent Visualization

In [ ]:
viz = EmbeddingVisualizer(inference_agent)

fig, ax = viz.plot_physics_scan(
    param_name="mass",
    scan_values=np.linspace(0.5, 2.0, 5, endpoint=False).tolist(),
    train_range=(0.5, 2.0),
    projection_method="tsne",
)
# fig.savefig("latent_mass_scan.png", dpi=150, bbox_inches="tight")

fig, ax = viz.plot_physics_scan(
    param_name="damping",
    scan_values=np.linspace(0.5, 2.0, 5, endpoint=False).tolist(),
    train_range=(0.5, 2.0),
    projection_method="tsne",
)

fig, ax = viz.plot_physics_scan(
    param_name="torque_scale",
    scan_values=np.linspace(0.3, 2.0, 8).tolist(),
    train_range=(0.5, 1.5),
    projection_method="tsne",
)
# fig.savefig("latent_torque_scan.png", dpi=150, bbox_inches="tight")

fig, ax = viz.plot_separation(num_trajs=10, projection_method="3d")
# fig.savefig("latent_separation.png", dpi=150, bbox_inches="tight")

viz.close()

### In distribution analysis

In [8]:
# In-distribution (ID) evaluation
#   5 dynamics configurations sampled within the training ranges
#   10 episodes per configuration -> 50 episodes
#   frozen policy and encoder (no updates; z is inferred from history only)


id_summary, id_per_config, id_df = evaluate_in_distribution(
    inference_agent, ENV_ID, n_configs=5, episodes_per_config=10, save_dir=checkpoint_path.parent)

display(id_per_config.round(3))
print(f"{ENV_ID} | In-distribution reward over {id_summary['n_episodes']} episodes: "
      f"{id_summary['mean']:.2f} ± {id_summary['std_episodes']:.2f} (std over episodes), "
      f"± {id_summary['std_configs']:.2f} (std over 5 configs)")

,mass_scale,damping_scale,torque_scale,friction_scale,reward_mean,reward_std,falls
config,,,,,,,
0,1.179,1.459,1.420,1.0,2385.352,505.603,10
1,1.023,0.830,0.882,1.0,3071.918,11.054,0
2,1.674,0.739,0.772,1.0,1426.973,829.162,7
3,1.132,1.033,1.226,1.0,2317.656,333.185,2
4,1.276,1.442,1.247,1.0,2971.111,583.857,7


Hopper-v5 | In-distribution reward over 50 episodes: 2434.60 ± 790.01 (std over episodes), ± 587.44 (std over 5 configs)


In [ ]:
ood_per_param, ood_per_config, ood_df, nominal_mean = evaluate_ood_single_param(
    inference_agent, ENV_ID, n_configs=5, episodes_per_config=10, margin=0.25,
    save_dir=checkpoint_path.parent)

print(f"{ENV_ID} | nominal reference reward: {nominal_mean:.2f}")
display(ood_per_config.round(3))
display(ood_per_param.round(2))
for param, r in ood_per_param.iterrows():
    print(f"{ENV_ID} | OOD {param:8s}: {r.reward_mean:8.2f} ± {r.std_episodes:7.2f} "
          f"(below {r.below_mean:.2f}, above {r.above_mean:.2f}, {r.pct_of_nominal:.0f}% of nominal, "
          f"{int(r.falls)}/{int(r.n_episodes)} falls)")